# Metabolomics VIP analyses

Date created: 12/22/2024

Notebook author: Yang Chen, Britta De Pessemier

Data analysis by: Britta De Pessemier, Yang Chen

This notebook plots the following:
- Top VIPs separating between pairwise skin groups and univariate statistical testing

In [1]:
# Import Python packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.ticker as ticker
from matplotlib.patches import Rectangle
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import Normalize


In [2]:
VIP_LvsH_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_LvsH_merged_data_method2_11212024.csv')
VIP_LvsH_merged['group'] = "H vs L"

VIP_NLvsH_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_NLvsH_merged_data_method2_11212024.csv')
VIP_NLvsH_merged['group'] = "H vs NL"

VIP_NLvsL_merged = pd.read_csv('../Data/metabolomics/Run3_10252024/output/Top_Features_VIP_NLvsL_merged_data_method2_11212024.csv')
VIP_NLvsL_merged['group'] = "NL vs L"

VIP_LvsH = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_LvsH_Load_data_method2_11212024.csv')
VIP_LvsH['group'] = "L vs H"

VIP_NLvsH = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_NLvsH_Load_data_method2_11212024.csv')
VIP_NLvsH['group'] = "NL vs H"

VIP_NLvsL = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIPs_NLvsL_Load_data_method2_11212024.csv')
VIP_NLvsL['group'] = "NL vs L"

# Replace values in the GroupContrib column
VIP_NLvsL_merged['GroupContrib'] = VIP_NLvsL_merged['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Acne Non-lesional': 'Acne_NL'
})
# Replace values in the GroupContrib column
VIP_NLvsL['GroupContrib'] = VIP_NLvsL['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Acne Non-lesional': 'Acne_NL'
})

VIP_NLvsH_merged['GroupContrib'] = VIP_NLvsH_merged['GroupContrib'].replace({
    'Acne Non-lesional': 'Acne_NL',
    'Healthy': 'Healthy'
})

VIP_NLvsH['GroupContrib'] = VIP_NLvsH['GroupContrib'].replace({
    'Acne Non-lesional': 'Acne_NL',
    'Healthy': 'Healthy'
})

VIP_LvsH_merged['GroupContrib'] = VIP_LvsH_merged['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Healthy': 'Healthy'
})

VIP_LvsH['GroupContrib'] = VIP_LvsH['GroupContrib'].replace({
    'Acne Lesional': 'Acne_L',
    'Healthy': 'Healthy'
})


In [3]:
# Concatenate the DataFrames vertically
VIP_combined = pd.concat([VIP_LvsH_merged, VIP_NLvsH_merged, VIP_NLvsL_merged], ignore_index=True)
#VIP_combined_Loadings = pd.concat([VIP_LvsH, VIP_NLvsH, VIP_NLvsL], ignore_index=True)

# Sort the combined DataFrame by the 'VIP' column in descending order
VIP_sorted = VIP_combined.sort_values(by='comp1', ascending=False)

# Replace NaN values in 'superclass', 'superclass', and 'superclass' columns with "Unknown"
VIP_sorted[['superclass', 'class', 'subclass']] = VIP_sorted[['superclass', 'class', 'subclass']].fillna("Unknown")

#VIP_sorted.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_all.csv')

# Filter out rows where VIP < 0.8
VIP_filtered = VIP_sorted[VIP_sorted['comp1'] >= 0.8]

# Filter out rows where 'molecular_formula' is NaN
VIP_filtered = VIP_filtered[VIP_filtered['molecular_formula'].notna()]


In [4]:
# Define a function to calculate the VIP_direction
def calculate_vip_direction(row):
    if row['group'] == 'H vs L':
        return row['comp1'] if row['GroupContrib'] == 'Acne_L' else -row['comp1']
    elif row['group'] == 'H vs NL':
        return row['comp1'] if row['GroupContrib'] == 'Acne_NL' else -row['comp1']
    elif row['group'] == 'NL vs L':
        return row['comp1'] if row['GroupContrib'] == 'Acne_L' else -row['comp1']
    else:
        return row['comp1']  # Default to keeping the value unchanged if no match

# Apply the function to create the new column
VIP_filtered['VIP_direction'] = VIP_filtered.apply(calculate_vip_direction, axis=1)

VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_data_method2_11212024.csv')
#VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final.csv')

In [5]:
# VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_name.csv')
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_data_method2_11212024.csv')

# Filter out rows where GroupContrib is NaN
VIP_filtered = VIP_filtered.dropna(subset=['GroupContrib'])
VIP_filtered.head()

,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,...,InChIKey.Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction
0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219
1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565
2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726
3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,...,SNICXCGAKADSCV,Organoheterocyclic compounds,Pyridines and derivatives,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338
4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,...,NWGKJDSIEKMTRX,Lipids and lipid-like molecules,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722


### Perform univariate statistical testing on above metabolites w/ top VIPs

In [6]:
# Read in metabolomics table
#metaB_table = pd.read_csv('../Data/metabolomics/Run3_10252024/data_sample_10282024.csv')

metaB_table = pd.read_csv('../Data/metabolomics/Run3_10252024/metabolomics_data_02072025_sample_name.csv')

# Set the 'SampleID' column as the index
metaB_table.set_index('SampleID', inplace=True)

# Convert columns in metaB_table to int (if applicable)
metaB_table.columns = metaB_table.columns.astype(int)

# Convert features_to_keep to int
features_to_keep = VIP_filtered['ID'].unique().astype(int)

# Subset metaB_table using the integer feature list
metaB_table_subset = metaB_table.loc[:, metaB_table.columns.isin(features_to_keep)]

metaB_table_subset

,954,1368,941,655,5062,777,2969,1119,5930,3496,...,1197,3794,29059,27758,23297,17958,19710,24392,20967,11925
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.0,970487.500,1942604.500,567510.060,1473068.10,...,62007.1100,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.0000,125692.3300
LAMI.RD308.D2.C2,1121916.40,1669351.50,2905.5796,3065.1714,19280.236,0.0,750870.100,1618142.800,442653.660,286004.00,...,87817.2000,5348.9400,231525.840,14299.3520,0.0000,0.000,0.000,0.00,3822.6377,4734.5693
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.0,595568.940,1605551.000,328151.120,299140.06,...,58588.9260,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.0,303845.300,865753.500,152392.720,191713.67,...,20153.3320,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.0,293386.340,1321474.400,137201.390,705570.80,...,45193.9400,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.0000,49464.9020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.0,579058.800,1380958.500,498893.780,236111.75,...,24563.9240,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.0000,0.0000
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.0,281896.700,742416.600,262886.470,303509.40,...,24480.3900,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.0000,220846.1600
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.0,255542.720,1194061.900,121803.890,5406879.00,...,35224.5800,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.0000,25695.0140


In [7]:
# Read in metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')

# Subset metadata based on the indices of metaB_subset
metadata_subset = metadata[metadata['SampleID'].isin(metaB_table_subset.index)]

metadata_subset

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
6,LAMI.RD317.D14.C2,C2,70,Lesional,skin,Day 14,44,14,317,15,...,44,70,25.042556,acne,RD317,acne,PP_317,PP_317C2,Lesional_C2,Acne_L
7,LAMI.RD305.D23.C1,C1,not applicable,Lesional,skin,Day 23,not applicable,23,305,not applicable,...,0,0,19.326616,acne,RD305,acne,PP_305,PP_305C1,Lesional_C1,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,LAMI.RD305.D0.C2,C2,69,Lesional,skin,Day 0,12,0,305,27,...,12,69,23.912250,acne,RD305,acne,PP_305,PP_305C2,Lesional_C2,Acne_L
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy


In [8]:
# Map the 'group' column from metadata_subset to metaB_subset
metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])

# Replace values in the 'group' column
metaB_table_subset['group'] = metaB_table_subset['group'].replace({
    'Healthy': 'H',
    'Acne_L': 'L',
    'Acne_NL': 'NL'
})

metaB_table_subset

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/2462488200.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/2462488200.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset['group'].replace({


,954,1368,941,655,5062,777,2969,1119,5930,3496,...,3794,29059,27758,23297,17958,19710,24392,20967,11925,group
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D0.C1,2987383.80,2025607.40,29574.6480,39859.7580,28277.594,0.0,970487.500,1942604.500,567510.060,1473068.10,...,12406.8260,611709.700,16778.8380,0.0000,0.000,0.000,0.00,0.0000,125692.3300,H
LAMI.RD308.D2.C2,1121916.40,1669351.50,2905.5796,3065.1714,19280.236,0.0,750870.100,1618142.800,442653.660,286004.00,...,5348.9400,231525.840,14299.3520,0.0000,0.000,0.000,0.00,3822.6377,4734.5693,L
LAMI.RD304.D11.C1,1818308.60,1434033.20,47379.2730,36750.4000,15391.807,0.0,595568.940,1605551.000,328151.120,299140.06,...,9403.6520,78491.530,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920,L
LAMI.RD304.D11.C2,1126503.20,706197.75,3460.0974,0.0000,15015.189,0.0,303845.300,865753.500,152392.720,191713.67,...,2286.2397,261996.800,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100,L
LAMI.RD304.D0.C1,827466.30,778566.10,4547.2640,3345.8184,45187.030,0.0,293386.340,1321474.400,137201.390,705570.80,...,0.0000,427202.060,3228.1143,3431.0034,0.000,0.000,0.00,0.0000,49464.9020,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1173415.00,1250164.60,8613.8590,9787.1210,14712.559,0.0,579058.800,1380958.500,498893.780,236111.75,...,5144.6616,122958.260,56654.0430,0.0000,0.000,0.000,0.00,0.0000,0.0000,NL
LAMI.RD308.D9.C3,550094.94,687527.06,3163.9185,3076.9688,17934.553,0.0,281896.700,742416.600,262886.470,303509.40,...,0.0000,65659.700,21826.1170,0.0000,0.000,0.000,0.00,0.0000,220846.1600,NL
LAMI.RD319.D2.C2,602147.10,538412.94,4006.6484,0.0000,10556.326,0.0,255542.720,1194061.900,121803.890,5406879.00,...,0.0000,73231.560,19741.2460,0.0000,0.000,0.000,0.00,0.0000,25695.0140,L


In [9]:
# Prepare the dictionary to set up univariate testing
feature_group_dict = (
    VIP_filtered.groupby('ID')['group']
    .apply(lambda x: x.unique().tolist())
    .to_dict()
)

In [10]:
# Initialize an empty dictionary to store results
statistical_results = {}

# Iterate through each metabolite in feature_group_dict
for metabolite, comparisons in feature_group_dict.items():
    # Create a dictionary to store results for the current metabolite
    metabolite_results = {}
    
    # Get the values for the metabolite from metaB_table_subset
    metabolite_values = metaB_table_subset[metabolite]
    
    # Iterate through each comparison for the current metabolite
    for comparison in comparisons:
        # Split the comparison into X and Y
        group_x, group_y = comparison.split(" vs ")
        
        # Subset values for the two groups
        values_x = metabolite_values[metaB_table_subset['group'] == group_x]
        values_y = metabolite_values[metaB_table_subset['group'] == group_y]
        
        # Perform the Mann-Whitney U test
        stat, p_value = mannwhitneyu(values_x, values_y, alternative='two-sided')
        
        # Store the result
        metabolite_results[comparison] = {'U-statistic': stat, 'p-value': p_value}
    
    # Add results for the current metabolite to the main results dictionary
    statistical_results[metabolite] = metabolite_results

# Convert results to a DataFrame for easier viewing (optional)
results_df = pd.DataFrame.from_dict({(met, comp): vals 
                                     for met, comps in statistical_results.items() 
                                     for comp, vals in comps.items()}, 
                                    orient='index')


# View results
results_df


U-statistic   p-value
379   NL vs L       2506.0  0.118156
      H vs NL        575.0  0.246680
655   NL vs L       2622.0  0.035492
      H vs NL        606.0  0.405200
777   H vs NL        876.5  0.027488
      NL vs L       2071.0  0.767129
      H vs L        2942.0  0.016249
862   NL vs L       2472.0  0.155851
      H vs NL        590.0  0.319225
941   H vs L        2218.0  0.526262
954   H vs NL        551.0  0.156106
      H vs L        2237.0  0.576631
1119  NL vs L       2486.0  0.139352
      H vs NL        639.0  0.641541
1197  NL vs L       2495.5  0.128945
      H vs NL        598.0  0.363119
1368  H vs NL        538.0  0.118988
      NL vs L       2558.0  0.074771
      H vs L        2222.0  0.536907
2969  H vs NL        531.0  0.102086
      NL vs L       2579.0  0.061424
      H vs L        2189.0  0.454638
3496  H vs NL        805.0  0.186551
3794  NL vs L       2513.0  0.109209
      H vs L        2275.0  0.680104
5062  H vs NL        626.0  0.544321
      H vs L        2367.0  0.963808
5930  NL vs L       2769.0  0.007546
      H vs NL        501.0  0.050040
      H vs L        2193.0  0.464216
7778  H vs NL        644.0  0.584706
      H vs L        2513.0  0.431753
11925 H vs NL        610.0  0.426629
      H vs L        2264.0  0.644373
15770 NL vs L       2142.0  0.963042
17958 NL vs L       2156.0  0.881355
19710 NL vs L       2136.0  0.997156
20967 H vs NL        625.0  0.224219
      NL vs L       2142.0  0.963042
23297 H vs L        1725.5  0.000763
      NL vs L       2404.5  0.177976
24392 NL vs L       1743.0  0.080255
      H vs NL        669.0  0.872994
27758 H vs NL        558.0  0.179420
      H vs L        2232.5  0.564559
29059 H vs NL        469.0  0.021106
      H vs L        1864.0  0.042352

In [11]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='ID')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)

# Create a mapping from Feature to Shortened_Compound_Name
#feature_to_name_mapping = VIP_filtered_unique.set_index('ID')['Shortened_Compound_Name']
feature_to_name_mapping = VIP_filtered_unique.set_index('ID')['Compound_Name']

# Map the Shortened_Compound_Name to results_df
#results_df['Shortened_Compound_Name'] = feature_index.map(feature_to_name_mapping)
results_df['Compound_Name'] = feature_index.map(feature_to_name_mapping)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/111036892.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)


In [12]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='ID')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)

# Create a mapping from Feature to VIP values
feature_to_vip_mapping = VIP_filtered_unique.set_index('ID')['comp1']

# Map the VIP values to results_df
results_df['comp1'] = feature_index.map(feature_to_vip_mapping)

# Sort results_df by the 'VIP' column in descending order
results_df = results_df.sort_values(by='comp1', ascending=False)
results_df = results_df.drop((777,), errors='ignore')

results_df

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/829313943.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/829313943.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  results_df = results_df.drop((777,), errors='ignore')


U-statistic   p-value  \
29059 H vs L        1864.0  0.042352   
      H vs NL        469.0  0.021106   
5930  H vs L        2193.0  0.464216   
      H vs NL        501.0  0.050040   
      NL vs L       2769.0  0.007546   
24392 H vs NL        669.0  0.872994   
      NL vs L       1743.0  0.080255   
3794  H vs L        2275.0  0.680104   
      NL vs L       2513.0  0.109209   
17958 NL vs L       2156.0  0.881355   
27758 H vs L        2232.5  0.564559   
      H vs NL        558.0  0.179420   
7778  H vs L        2513.0  0.431753   
      H vs NL        644.0  0.584706   
2969  H vs NL        531.0  0.102086   
      NL vs L       2579.0  0.061424   
      H vs L        2189.0  0.454638   
1368  H vs NL        538.0  0.118988   
      NL vs L       2558.0  0.074771   
      H vs L        2222.0  0.536907   
655   H vs NL        606.0  0.405200   
      NL vs L       2622.0  0.035492   
15770 NL vs L       2142.0  0.963042   
379   H vs NL        575.0  0.246680   
      NL vs L       2506.0  0.118156   
954   H vs L        2237.0  0.576631   
      H vs NL        551.0  0.156106   
1197  H vs NL        598.0  0.363119   
      NL vs L       2495.5  0.128945   
862   NL vs L       2472.0  0.155851   
      H vs NL        590.0  0.319225   
20967 H vs NL        625.0  0.224219   
      NL vs L       2142.0  0.963042   
5062  H vs L        2367.0  0.963808   
      H vs NL        626.0  0.544321   
23297 H vs L        1725.5  0.000763   
      NL vs L       2404.5  0.177976   
1119  H vs NL        639.0  0.641541   
      NL vs L       2486.0  0.139352   
11925 H vs L        2264.0  0.644373   
      H vs NL        610.0  0.426629   
941   H vs L        2218.0  0.526262   
19710 NL vs L       2136.0  0.997156   
3496  H vs NL        805.0  0.186551   

                                                   Compound_Name     comp1  
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV  2.271726  
      H vs NL                             D-TRYPTOPHAN - 60.0 eV  2.271726  
      NL vs L                             D-TRYPTOPHAN - 60.0 eV  2.271726  
24392 H vs NL  Sorbitane Monooleate - Polysorbate 20 in-sourc...  1.947722  
      NL vs L  Sorbitane Monooleate - Polysorbate 20 in-sourc...  1.947722  
3794  H vs L                  Massbank:PR311057 Glutamyltyrosine  1.824152  
      NL vs L                 Massbank:PR311057 Glutamyltyrosine  1.824152  
17958 NL vs L                                         Sinensetin  1.823974  
27758 H vs L   Spectral Match to N-Oleoylethanolamine from NI...  1.814812  
      H vs NL  Spectral Match to N-Oleoylethanolamine from NI...  1.814812  
7778  H vs L                   Phenylbenzimidazole sulfonic acid  1.729160  
      H vs NL                  Phenylbenzimidazole sulfonic acid  1.729160  
2969  H vs NL                           Phenylalanine - 50.00 eV  1.701639  
      NL vs L                           Phenylalanine - 50.00 eV  1.701639  
      H vs L                            Phenylalanine - 50.00 eV  1.701639  
1368  H vs NL                               ISOLEUCINE - 60.0 eV  1.691869  
      NL vs L                               ISOLEUCINE - 60.0 eV  1.691869  
      H vs L                                ISOLEUCINE - 60.0 eV  1.691869  
655   H vs NL                                      UROCANIC ACID  1.662492  
      NL vs L                                      UROCANIC ACID  1.662492  
15770 NL vs L                                     xylometazoline  1.615550  
379   H vs NL                                            URIDINE  1.520722  
      NL vs L                                            URIDINE  1.520722  
954   H vs L                       L-Pyroglutamic acid - 40.0 eV  1.501429  
      H vs NL                      L-Pyroglutamic acid - 40.0 eV  1.501429  
1197  H vs NL           Massbank:PR310825 N-Fructosyl isol

In [13]:
VIP_filtered_unique

,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,...,InChIKey.Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction
0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219
2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726
3,20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,...,SNICXCGAKADSCV,Organoheterocyclic compounds,Pyridines and derivatives,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338
4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,...,NWGKJDSIEKMTRX,Lipids and lipid-like molecules,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722
6,49,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,306400,...,VVLXCWVSSLFQDS,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Dipeptides,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005747606,NL vs L,-1.824152
7,50,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,637600,...,LKMNXYDUQXAUCZ,Phenylpropanoids and polyketides,Flavonoids,O-methylated flavonoids,Flavonoids,Flavones,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012283450,NL vs L,1.823974
8,24,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,282600,...,BOWVQLFMWHZBEF,Organic nitrogen compounds,Organonitrogen compounds,Amines,Fatty amides,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,1.814812
10,25,7778,1.729160,Acne_NL,-0.051577,CCMSLIB00000579531,spectra_filtered.mgf,CASMI.mgf,0.968451,17502000,...,UVCJGUGAGLDPAA,Organoheterocyclic compounds,Benzimidazoles,Phenylbenzimidazoles,NaN,NaN,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000579531,H vs NL,1.729160
12,27,2969,1.701639,Acne_NL,-0.050756,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,...,COLNVLDHVKWLRT,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,H vs NL,1.701639
13,28,1368,1.691869,Acne_NL,-0.050464,CCMSLIB00005883686,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.992042,12116000,...,AGPKZVBTJJNPAG,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883686,H vs NL,1.691869


In [14]:
# Filter rows where the p-value is greater than 0.1
filtered_features = results_df[results_df['p-value'] <= 0.10]

# Add shortened names of compounds for plotting purposes
shortened_names = ["C19H36O4 Fatty alcohol", "C19H36O4 Fatty alcohol", "Tryptophan", "Tryptophan", "Sorbitane Monooleate", "Phenylalanine", "Isoleucine", "Urocanic acid", "N-acyl lipid glutamine-C14:0"]  # your list of names

# Also add ID column for later mapping
feature_ids = [29059, 29059, 5930, 5930, 24392, 2969, 1368, 655, 23297]  # your list of names

# Add lists as columns
filtered_features['Shortened_Compound_Name'] = shortened_names
filtered_features['ID'] = feature_ids

filtered_features

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/884944378.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_features['Shortened_Compound_Name'] = shortened_names
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/884944378.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_features['ID'] = feature_ids


U-statistic   p-value  \
29059 H vs L        1864.0  0.042352   
      H vs NL        469.0  0.021106   
5930  H vs NL        501.0  0.050040   
      NL vs L       2769.0  0.007546   
24392 NL vs L       1743.0  0.080255   
2969  NL vs L       2579.0  0.061424   
1368  NL vs L       2558.0  0.074771   
655   NL vs L       2622.0  0.035492   
23297 H vs L        1725.5  0.000763   

                                                   Compound_Name     comp1  \
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219   
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219   
5930  H vs NL                             D-TRYPTOPHAN - 60.0 eV  2.271726   
      NL vs L                             D-TRYPTOPHAN - 60.0 eV  2.271726   
24392 NL vs L  Sorbitane Monooleate - Polysorbate 20 in-sourc...  1.947722   
2969  NL vs L                           Phenylalanine - 50.00 eV  1.701639   
1368  NL vs L                               ISOLEUCINE - 60.0 eV  1.691869   
655   NL vs L                                      UROCANIC ACID  1.662492   
23297 H vs L                                           Gln-C14:0  1.332919   

                    Shortened_Compound_Name     ID  
29059 H vs L         C19H36O4 Fatty alcohol  29059  
      H vs NL        C19H36O4 Fatty alcohol  29059  
5930  H vs NL                    Tryptophan   5930  
      NL vs L                    Tryptophan   5930  
24392 NL vs L          Sorbitane Monooleate  24392  
2969  NL vs L                 Phenylalanine   2969  
1368  NL vs L                    Isoleucine   1368  
655   NL vs L                 Urocanic acid    655  
23297 H vs L   N-acyl lipid glutamine-C14:0  23297

In [15]:
# Group by the primary index and collect the secondary index values
keep_dict = filtered_features.groupby(level=0).apply(lambda df: df.index.get_level_values(1).tolist()).to_dict()

keep_dict

{655: ['NL vs L'],
 1368: ['NL vs L'],
 2969: ['NL vs L'],
 5930: ['H vs NL', 'NL vs L'],
 23297: ['H vs L'],
 24392: ['NL vs L'],
 29059: ['H vs L', 'H vs NL']}

In [16]:
# Function to check if the Feature and group combination is in keep_dict
def filter_rows(row):
    feature = row['ID']
    group = row['group']
    return feature in keep_dict and group in keep_dict[feature]

# Apply the filter function to VIP_filtered
VIP_filtered_subset = VIP_filtered[VIP_filtered.apply(filter_rows, axis=1)]

VIP_filtered_subset

,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,...,InChIKey.Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction
0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219
1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565
2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726
4,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,...,NWGKJDSIEKMTRX,Lipids and lipid-like molecules,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722
5,23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,1.922795
14,53,655,1.662492,Acne_L,0.049478,CCMSLIB00006681721,spectra_filtered.mgf,MONA.mgf,0.804832,4820000,...,LOIYMIARKYCTBW,Organoheterocyclic compounds,Azoles,Imidazoles,NaN,NaN,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00006681721,NL vs L,1.662492
18,57,2969,1.509916,Acne_NL,-0.044937,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,...,COLNVLDHVKWLRT,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,NL vs L,-1.509916
23,60,1368,1.395686,Acne_NL,-0.041538,CCMSLIB00005883686,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.992042,12116000,...,AGPKZVBTJJNPAG,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883686,NL vs L,-1.395686
29,6,23297,1.332919,Healthy,-0.039652,CCMSLIB00011435350,spectra_filtered.mgf,ECG-ACYL-AMIDES-C4-C24-LIBRARY.mgf,0.789557,348900,...,GQCSFCKGCIEEJN,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Fatty amides,N-acyl amines,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00011435350,H vs L,-1.332919


In [17]:
# Ensure Feature in VIP_filtered_subset and the primary index in filtered_features are the same type
VIP_filtered_subset['ID'] = VIP_filtered_subset['ID'].astype(int)
filtered_features.index = filtered_features.index.set_levels(
    filtered_features.index.levels[0].astype(int), level=0
)

# Create a MultiIndex mapping of (Feature, group) to p-value
p_value_mapping = filtered_features['p-value']

# Create a new column in VIP_filtered_subset for the p_value_bh
VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['ID', 'group']).index.map(p_value_mapping)

# Reset the index to return to the original structure if needed
VIP_filtered_subset.reset_index(drop=True, inplace=True)

# Map the name from Shortened_Compound_Name
# Drop duplicate IDs, keeping the first occurrence
unique_mapping = filtered_features.drop_duplicates(subset='ID').set_index('ID')['Shortened_Compound_Name']

# Now map the values
VIP_filtered_subset['Shortened_Compound_Name'] = VIP_filtered_subset['ID'].map(unique_mapping)
VIP_filtered_subset.loc[VIP_filtered_subset['Shortened_Compound_Name'] == 'Tryptophan', 'subclass'] = 'Amino acids, peptides, and analogues'


# View the updated DataFrame
VIP_filtered_subset


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/1657808842.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['ID'] = VIP_filtered_subset['ID'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/1657808842.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['ID', 'group']).index.map(p_value_mapping)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/1657808842.py:21:

,Unnamed: 0,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,...,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group,VIP_direction,p-value,Shortened_Compound_Name
0,18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,0.021106,C19H36O4 Fatty alcohol
1,0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,...,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,0.042352,C19H36O4 Fatty alcohol
2,45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,Indoles and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,0.007546,Tryptophan
3,48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,...,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,0.080255,Sorbitane Monooleate
4,23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,...,Indoles and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,1.922795,0.050040,Tryptophan
5,53,655,1.662492,Acne_L,0.049478,CCMSLIB00006681721,spectra_filtered.mgf,MONA.mgf,0.804832,4820000,...,Azoles,Imidazoles,NaN,NaN,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00006681721,NL vs L,1.662492,0.035492,Urocanic acid
6,57,2969,1.509916,Acne_NL,-0.044937,CCMSLIB00005885034,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.971869,11500000,...,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005885034,NL vs L,-1.509916,0.061424,Phenylalanine
7,60,1368,1.395686,Acne_NL,-0.041538,CCMSLIB00005883686,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.992042,12116000,...,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883686,NL vs L,-1.395686,0.074771,Isoleucine
8,6,23297,1.332919,Healthy,-0.039652,CCMSLIB00011435350,spectra_filtered.mgf,ECG-ACYL-AMIDES-C4-C24-LIBRARY.mgf,0.789557,348900,...,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Fatty amides,N-acyl amines,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00011435350,H vs L,-1.332919,0.000763,N-acyl lipid glutamine-C14:0


In [18]:
VIP_filtered_subset['subclass'].unique()

array(['Fatty alcohols', 'Amino acids, peptides, and analogues',
       'Fatty acid esters', 'Imidazoles'], dtype=object)

In [19]:
# Create a 1x3 subplot with gridspec_kw for spacing
fig, axes = plt.subplots(
    1, 3, figsize=(22, 6), sharey=False,  # Increase width to make space for heatmaps 22 , 6
    gridspec_kw={"wspace": 0.75}  # Adjust wspace as needed
)

unique_groups = ["H vs L", "H vs NL", "NL vs L"]  # Explicitly set the order of groups


color_map = {
    'Fatty acid esters': '#8B4000',  # Bright orange,
    'Fatty alcohols': '#F28C28', # Dark orange
    'Amino acids, peptides, and analogues': '#18becf',  # Bright blue
    'Imidazoles': '#BF40BF',  # Purple
}

# Add an overall title to the figure
fig.suptitle('Top Driving Metabolites Features Across Groups', fontsize=20, y=1.025)

# Loop through each group and plot in the corresponding subplot
for i, group in enumerate(unique_groups):
    # Filter data for the current group
    group_data = VIP_filtered_subset[VIP_filtered_subset['group'] == group]
    vip_scores = group_data['VIP_direction']
    metabolites = group_data['Shortened_Compound_Name']
    # metabolites = group_data['Compound_Name']
    classes = group_data['subclass']
    p_values = group_data['p-value']  # Use raw p-values directly

    # Map colors for the current group
    dot_colors = classes.map(color_map).fillna('#808080')

    # Format y-tick labels to add a newline after each word
    formatted_metabolites = [label.replace(' ', '\n') for label in metabolites]

    # Plot scatter plot on the left
    scatter = axes[i].scatter(
        vip_scores,
        range(len(group_data)),
        color=dot_colors,
        s=200  # Adjust the size of the dots
    )
    axes[i].axvline(x=1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=1
    axes[i].axvline(x=-1, color='gray', linestyle='--', linewidth=1.5)  # Add vertical dashed line at x=-1

    # Set y-ticks and y-tick labels for scatter plot
    axes[i].set_yticks(range(len(metabolites)))
    axes[i].set_yticklabels(formatted_metabolites, fontsize=12)
    axes[i].tick_params(axis='y', labelleft=True)
    axes[i].invert_yaxis()
    axes[i].set_xlabel('VIP Scores', fontsize=12)

    # Define custom titles for each group
    custom_titles = {
        "H vs L": "Healthy (left) vs Lesional (right)",
        "H vs NL": "Healthy (left) vs Non-lesional (right)",
        "NL vs L": "Non-lesional (left) vs Lesional (right)"
    }

    axes[i].set_title(custom_titles.get(group, f"({group})"), fontsize=16, pad=10)

    # Add heatmap to the right of each subplot
    ax_heatmap = fig.add_axes([axes[i].get_position().x1 + 0.02,  # Position heatmap to the right of scatter
                               axes[i].get_position().y0,
                               0.01,
                               axes[i].get_position().height])
    
    # Use raw p-values for visualization
    p_values_array = p_values.values.reshape(-1, 1)  # Convert to 2D array for heatmap

    # Generate tick positions for the colorbar
    p_value_min = VIP_filtered_subset['p-value'].min()
    p_value_max = VIP_filtered_subset['p-value'].max()

    # Normalize the color scale for the heatmap
    norm = Normalize(vmin=p_value_min, vmax=p_value_max)

    heatmap = ax_heatmap.imshow(
        p_values_array,
        norm=norm,
        cmap=sns.color_palette("Blues_r", as_cmap=True),  # Use a color map for the heatmap
        aspect="auto",
        origin="lower"
    )

    # Set heatmap ticks to match scatter plot
    ax_heatmap.set_yticks(range(len(metabolites)))
    ax_heatmap.set_yticklabels([])  # Hide y-tick labels for heatmap
    ax_heatmap.tick_params(axis='y', length=0)  # Remove tick marks
    ax_heatmap.set_xticks([])  # Hide x-tick labels
    ax_heatmap.set_title('p-value', fontsize=12)

# Add a single legend at the bottom
legend_handles = [plt.Line2D([0], [0], marker='o', color=color, linestyle='', markersize=12, label=subclass_)
                  for subclass_, color in color_map.items()]
fig.legend(handles=legend_handles, title='Subclass category', loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize=12, title_fontsize=14, labelspacing=1.5, borderaxespad=-3)

# Add a colorbar for the heatmaps
cbar_ax = fig.add_axes([0.35, -0.3, 0.3, 0.02])  # Position for colorbar at the bottom
cbar = plt.colorbar(heatmap, cax=cbar_ax, orientation='horizontal')

# Use np.linspace to generate evenly spaced ticks
tick_positions = np.linspace(p_value_min, p_value_max, num=5)
print(tick_positions)

# Format tick labels as plain decimal values
tick_labels = [f"{tick:.6f}" for tick in tick_positions]  # Adjust the number of decimal places as needed
print(tick_labels)

# Set colorbar ticks and labels
cbar.set_ticks(tick_positions)
cbar.set_ticklabels(tick_labels, fontsize=12)
cbar.set_label('p-value', fontsize=14, labelpad=20)
cbar.ax.xaxis.set_label_position('top')

# Adjust layout and save the figure
plt.tight_layout(rect=[0, 0.1, 0.9, 1])  # Adjust layout to leave space for the legend and colorbar

output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot_with_pvalue_heatmaps.svg"
plt.savefig(output_path)

output_path = "../Figures/metabolomics_Figures/analysis/VIP_combined_plot_with_pvalue_heatmaps.png"
plt.savefig(output_path, dpi=600, bbox_inches='tight')


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_45193/2194501041.py:117: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.1, 0.9, 1])  # Adjust layout to leave space for the legend and colorbar


[0.00076289 0.02063602 0.04050915 0.06038227 0.0802554 ]
['0.000763', '0.020636', '0.040509', '0.060382', '0.080255']
